## Project 4. Bagdanova A.B.

* Цель: Подготовить основу рекомендательной системы онлайн-школы по продаже образовательных курсов

Ниже приведены коды SQL и ответы, которые они выводят (если нажать на редактирование кода, то он обретёт стандартный вид столбиком)

with base as
(
SELECT 
c.user_id,
c.purchased_at,
c.id as cart_id,
c.state,
ci.resource_type,
ci.resource_id,
ci.cart_id as items_id,
ci.id as operation_id
FROM final.carts c 
join final.cart_items ci on c.id=ci.cart_id
)
SELECT 
COUNT(DISTINCT user_id) AS total_clients
FROM base
WHERE resource_type = 'Course' AND state = 'successful'

* 49006 клиентов купили курсы

with base as
(
SELECT 
c.user_id,
c.purchased_at,
c.id as cart_id,
c.state,
ci.resource_type,
ci.resource_id,
ci.cart_id as items_id,
ci.id as operation_id
FROM final.carts c 
join final.cart_items ci on c.id=ci.cart_id
)
SELECT 
COUNT(DISTINCT resource_id) AS distinct_resourse_id
FROM base
WHERE resource_type = 'Course'

* Есть 127 различных курсов

with base as
(
SELECT 
c.user_id,
c.purchased_at,
c.id as cart_id,
c.state,
ci.resource_type,
ci.resource_id,
ci.cart_id as items_id,
ci.id as operation_id
FROM final.carts c 
join final.cart_items ci on c.id=ci.cart_id
)
SELECT AVG(courses_count) AS avg_courses_per_client
FROM (
    SELECT user_id, COUNT(resource_id) AS courses_count
    FROM base
    WHERE resource_type = 'Course'
      AND state = 'successful'
    GROUP BY user_id
) AS user_courses;

* Среднее число купленных курсов на одного клиента: 1.44 

with base as
(
SELECT 
c.user_id,
c.purchased_at,
c.id as cart_id,
c.state,
ci.resource_type,
ci.resource_id,
ci.cart_id as items_id,
ci.id as operation_id
FROM final.carts c 
join final.cart_items ci on c.id=ci.cart_id
)
SELECT COUNT(*) AS users_with_more_than_one_course
FROM (
    SELECT user_id
    FROM base
    WHERE resource_type = 'Course' AND state = 'successful'
    GROUP BY user_id
    HAVING COUNT(DISTINCT resource_id) > 1
) AS with_courses;


* 12656 пользователей купили больше одного курса

## Анализ полученных данных

In [1]:
import pandas as pd
import itertools
import numpy as np

In [2]:
df = pd.read_csv(r'C:\Users\днс\Downloads\courses.csv')

In [3]:
# Группируем по клиентам и удаляем дубликаты курсов у user_id

pairs_df = df.groupby('user_id')['resource_id'].apply(lambda x:list(np.unique(x))).reset_index()

In [4]:
# Фильтрация клиентов, у которых курсов больше 2

pairs_df = pairs_df[pairs_df['resource_id'].apply(len)>=2]

In [5]:
# Генерируем уникальные пары в виде кортежей

courses = []
for lst in pairs_df['resource_id']:
    for x, y in itertools.combinations(lst,2):
        courses.append(tuple(sorted((x, y))))

In [6]:
courses_df = pd.Series(courses)

In [7]:
print(f"Уникальных комбинаций: {len(courses_df.value_counts())}")

Уникальных комбинаций: 3989


In [ ]:
courses_df.value_counts()

(551, 566)     797
(515, 551)     417
(489, 551)     311
(523, 551)     304
(566, 794)     290
              ... 
(800, 1144)      1
(489, 1185)      1
(765, 814)       1
(745, 814)       1
(571, 814)       1
Name: count, Length: 3989, dtype: int64

* 551 и 566 самая популярная пара курсов

## Таблица рекомендаций

In [9]:
from collections import defaultdict, Counter

* Отфильтруем пары курсов и установим минимальную границу "50"

In [10]:
pair_counts = courses_df.value_counts().reset_index()
pair_counts.columns = ['pair', 'count']
significant_pairs = pair_counts[pair_counts['count'] >= 50]
print(significant_pairs)

           pair  count
0    (551, 566)    797
1    (515, 551)    417
2    (489, 551)    311
3    (523, 551)    304
4    (566, 794)    290
..          ...    ...
143  (551, 776)     51
144  (514, 809)     50
145  (551, 659)     50
146  (566, 752)     50
147  (564, 809)     50

[148 rows x 2 columns]


In [11]:
# Создадим словарь для курсов и список пар с ним

recommendations = defaultdict(list)

for (course1, course2), count in courses_df.value_counts().items():
    recommendations[course1].append((course2, count))
    recommendations[course2].append((course1, count))

# Самые поппулярные курсы для замены

popularity = Counter()
for (c1, c2), count in courses_df.value_counts().items():
    popularity[c1] += count
    popularity[c2] += count

top1 = popularity.most_common(1)[0][0] 
top2 = popularity.most_common(2)[1][0] if len(popularity) >= 2 else top1

# Минимальная граница
minimal = 50 
result = []

for course, recs in recommendations.items():
    sorted_recs = sorted(recs, key=lambda x: x[1], reverse=True)
    
    # Рекомендация №1
    
    if len(sorted_recs) > 0 and sorted_recs[0][1] >= minimal:
        rec1 = sorted_recs[0][0]
    else:
        rec1 = top1

    # Рекомендация №2

    if len(sorted_recs) > 1 and sorted_recs[1][1] >= minimal:
        rec2 = sorted_recs[1][0]
    else: 
        if top2 != rec1:
           rec2 = top2
        else:
            rect2 = top1
        
    result.append([course, rec1, rec2])

In [12]:
# Сохраняем данные

final_df = pd.DataFrame(result, columns=['Курс', 'Рекомендация №1', 'Рекомендация №2'])
final_df = final_df.sort_values('Курс')
final_df.to_csv('recommendations.csv', index=False)

final_df.head()

,Курс,Рекомендация №1,Рекомендация №2
28,356,571,357
23,357,571,356
80,358,551,566
95,359,551,566
93,360,551,566


## Выводы

В ходе анализа частоты встречаемости пар курсов было выявлено, что большинство комбинаций являются единичными или случайными покупками. Такие пары не отражают устойчивый спрос и создают "шум" в рекомендательной системе. Чтобы исключить случайные срабатывания и оставить только статистически значимые связи, в качестве минимального порога было выбрано значение 50 совместных покупок.

Это позволяет:

1. Отсечь редкие, случайные комбинации;

2. Сосредоточиться на устойчивых паттернах покупательского поведения;

3. Обеспечить достаточный объем данных для формирования точных рекомендаций.

Если пара курсов не превышает порог в 50, она считается недостаточно популярной, и вместо нее в рекомендации подставляется глобально популярный курс (top1 или top2), что гарантирует релевантность итоговой таблицы.